# Tech Department Agent - RAG + Memory Test

This notebook tests the Streamlit tech department agent implementation.

It verifies:

- Tech department policies are ingested into ChromaDB
- RAG retrieval returns relevant policy sections
- Agent commands work
- Device request tickets are created
- Device issue tickets are created and prioritized
- Interactions are stored as episodic memory
- Agent facts and operating rules are stored as declarative memory

The test uses a temporary database directory, so it does not modify the Streamlit app's real `.tech_agent_memory/` data.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from tech_agent import DEFAULT_KB_PATH, OllamaClient, TechDepartmentAgent, TechMemoryRAG

print(f"Policy file: {DEFAULT_KB_PATH}")
print(f"Policy file exists: {DEFAULT_KB_PATH.exists()}")
assert DEFAULT_KB_PATH.exists(), "Missing tech department policy file"

## 1. Initialize Agent With Temporary Vector Memory

In [ ]:
temp_dir = TemporaryDirectory(ignore_cleanup_errors=True)
test_db_dir = Path(temp_dir.name) / "tech_agent_test_db"

rag = TechMemoryRAG(db_dir=test_db_dir, kb_path=DEFAULT_KB_PATH)
llm = OllamaClient(model_name="mistral")
agent = TechDepartmentAgent(rag=rag, llm=llm)

stats = rag.stats()
print(stats)
assert stats["policies"] >= 5, "Expected policy sections to be ingested"
assert stats["episodic_memories"] == 0, "Temporary episodic memory should start empty"
assert stats["declarative_memories"] >= 1, "Agent should start with declarative operating memory"
assert stats["tickets"] == 0, "Temporary test ticket store should start empty"

## 2. Test Policy RAG Retrieval

In [ ]:
results = rag.search("How much is the laptop budget and how long does approval take?", include_memory=False)

for item in results[:3]:
    print(f"{item.kind} | {item.similarity:.2f} | {item.id}")
    print(item.content[:300], "\n")

combined = "\n".join(item.content.lower() for item in results)
assert "laptop" in combined, "RAG should retrieve laptop policy context"
assert "2000" in combined or "5 business days" in combined, "RAG should retrieve budget or timeline details"

## 3. Test Commands

In [ ]:
help_result = agent.answer("/help")
stats_result = agent.answer("/stats")
policy_result = agent.answer("/policy remote work equipment")
remember_result = agent.answer("/remember Always ask for the asset tag before troubleshooting company laptops.")

print(help_result["answer"])
print(stats_result["answer"])
print(policy_result["answer"][:500])
print(remember_result["answer"])

assert "/policy" in help_result["answer"], "Help command should list policy command"
assert "policy sections" in stats_result["answer"], "Stats command should report policy count"
assert len(policy_result["sources"]) > 0, "Policy command should return retrieved sources"
assert "Declarative memory saved" in remember_result["answer"], "Remember command should save declarative memory"

## 4. Test Device Request Ticket

In [ ]:
request_result = agent.submit_device_request(
    {
        "employee": "Aisha Khan",
        "department": "Engineering",
        "manager": "Ravi Mehta",
        "device": "Laptop",
        "work_mode": "Hybrid",
        "business_reason": "Needs a development laptop for backend work",
        "preferred_date": "2026-09-20",
    }
)

print(request_result)
assert request_result["ticket_id"].startswith("TECH-"), "Device request should return a TECH ticket ID"
assert "Device request created" in request_result["answer"], "Device request answer should confirm creation"

## 5. Test Device Issue Ticket and Priority Classification

In [ ]:
issue_result = agent.submit_device_issue(
    {
        "employee": "Aisha Khan",
        "device_asset": "LT-ENG-1042",
        "urgency": "High",
        "issue_summary": "VPN is blocked before a customer meeting",
        "troubleshooting": "Restarted laptop and VPN client",
    }
)

print(issue_result)
assert issue_result["ticket_id"].startswith("TECH-"), "Device issue should return a TECH ticket ID"
assert issue_result["priority"] == "high", "VPN or meeting issue should be classified as high priority"

## 6. Test Memory and Ticket Retrieval

In [ ]:
final_stats = rag.stats()
print(final_stats)

memory_results = rag.search("Aisha VPN laptop engineering", include_memory=True)
for item in memory_results[:5]:
    print(f"{item.kind} | {item.similarity:.2f} | {item.id}")
    print(item.content[:250], "\n")

assert final_stats["tickets"] == 2, "Expected one request ticket and one issue ticket"
assert final_stats["episodic_memories"] >= 5, "Commands and ticket actions should create episodic memory entries"
assert final_stats["declarative_memories"] >= 2, "Seeded and user-taught facts should create declarative memory"
assert any(item.kind == "ticket" for item in memory_results), "Memory search should retrieve created tickets"
assert any(item.kind == "declarative" for item in memory_results), "Memory search should retrieve declarative memory"

## 7. Optional Ollama Answer Test

If Ollama with `mistral` is running, this cell checks a real generated answer. If Ollama is not available, the agent uses its policy-grounded fallback answer.

In [ ]:
answer_result = agent.answer("Can I get a new laptop for hybrid work and what details do you need?")

print(answer_result["answer"])
print(f"Sources returned: {len(answer_result['sources'])}")

assert len(answer_result["answer"]) > 50, "Agent should produce a useful answer"
assert len(answer_result["sources"]) > 0, "Agent should retrieve policy or memory context"

temp_dir.cleanup()
print("All tech department agent tests passed.")